<a href="https://colab.research.google.com/github/KalinaMarkova/Machine-Learning-Course-Project/blob/main/03_Model_Training_Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Grained Analysis of Propaganda in News Articles
## Notebook 03: Model Training for End-to-End Joint Tagger (29 Labels) - Experiment 1

In this notebook, we will load our tokenized and aligned dataset and use it to train a **RoBERTa** model to identify the exact boundaries of propaganda techniques by Token Classification.

We are going to add a brand-new, untrained mathematical layer directly on top of RoBERTa which will be the Token Classifier. As RoBERTa reads an article, it will push its contextual understanding of every single token up into that classification head. The head then will produce a prediction for every single token, trying to guess which of the integer labels (0 for non-propaganda and labels 1 through 28 for the specific propaganda categories) belongs to that token.

In [12]:
!pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e9864d93f37085f909bfd393ae6baa31f78fc45a7ced256ca9df292e1c4c8842
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [13]:
import torch
import numpy as np
import evaluate
from google.colab import drive
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments
)
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight




Let's load the dataset, split it and recreate again 29-class Label Dictionaries.

In [19]:
# 1. Load the dataset from your correct Drive path
dataset_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/exp1_joint_29labels_sentence_dataset'
dataset = load_from_disk(dataset_path)

# 2. Perform the Three-Way Split (80% Train, 10% Val, 10% Test)
# First, separate 80% for training and 20% for the temporary hold-out
train_temp_split = dataset.train_test_split(test_size=0.20, seed=42)
train_dataset = train_temp_split['train']
temp_dataset = train_temp_split['test']

# Next, split that 20% hold-out evenly into 10% Validation and 10% Test
val_test_split = temp_dataset.train_test_split(test_size=0.50, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']

# 3. Recreate the 29-class Label Dictionaries perfectly
semeval_techniques = [
    "Appeal_to_Authority", "Appeal_to_fear-prejudice", "Bandwagon,Reductio_ad_hitlerum",
    "Black-and-White_Fallacy", "Causal_Oversimplification", "Doubt",
    "Exaggeration,Minimisation", "Flag-Waving", "Loaded_Language",
    "Name_Calling,Labeling", "Repetition", "Slogans",
    "Thought-terminating_Cliches", "Whataboutism,Straw_Man,Red_Herring"
]

exp1_labels_list = ['O'] + [f"B-{t}" for t in semeval_techniques] + [f"I-{t}" for t in semeval_techniques]
exp1_labels_list = sorted(list(set(exp1_labels_list)))
exp1_labels_list.remove('O')
exp1_labels_list = ['O'] + exp1_labels_list

label2id = {label: i for i, label in enumerate(exp1_labels_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"Dataset split is completed.")
print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

Dataset split is completed.
Train size: 12022
Validation size: 1503
Test size: 1503


In order to manage the extreme class imbalance of non-propaganda text vs. the different types of propaganda and to avoid the model just guessing "O" for every word, we will use the `compute_class_weight` for the loss function of the model which mathematically penalizes the model for missing rare classes using the following Inverse Frequency equation:

$$W_j = \frac{N}{k \times n_j}$$

**Where:**
* $W_j$ = The final computed weight for class $j$
* $N$ = The total number of tokens (samples) in your entire training dataset
* $k$ = The total number of unique classes (e.g., 29)
* $n_j$ = The exact number of times class $j$ actually appears in the dataset



In [20]:
# Extract all labels from the training set and flatten them into a single list
all_train_labels = [label for sequence in train_dataset['labels'] for label in sequence]

# Identify unique classes present in the training set
unique_classes = np.unique(all_train_labels)

# Compute balanced class weights
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=all_train_labels)

# Create a full weight array for all 29 labels (defaulting to 1.0 if a class is completely missing)
class_weights = np.ones(len(label2id))
for i, cls in enumerate(unique_classes):
    class_weights[cls] = weights[i]

# Convert to a PyTorch Tensor and move it to your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Class weights successfully calculated for {len(unique_classes)} unique classes!")
print(f"Hardware in use: {device}")

Class weights successfully calculated for 29 unique classes!
Hardware in use: cuda


RoBERTa was originally trained to fill in missing words. By using `AutoModelForTokenClassification`, Hugging Face removes RoBERTa's original "guessing" head and glues on a brand-new, completely blank neural network layer designed specifically to output one of your 29 labels (`num_labels`). When `num_labels = 29`, this projection compresses the 768 complex features representing a token down into **29 raw numbers**—one scalar confidence score for every category in your label dictionary.

For example, the output array of raw logits for the word *"fake"* might look like this:

| Label Index | Label String (`id2label`) | Raw Logit Score (Model Confidence) |
| :--- | :--- | :--- |
| `0` | `"O"` | `-1.2` |
| `1` | `"B-Loaded_Language"` | **`+8.5`** $\leftarrow$ *(Highest Score)* |
| `2` | `"I-Loaded_Language"` | `+0.3` |
| `3` | `"B-Name_Calling,Labeling"` | `+1.1` |
| `...` | `...` | `...` |
| `28` | `"I-Slogans"` | `-3.4` |



`weight_decay=0.01`: A regularization technique that slightly shrinks the model's weights during training. This prevents the model from memorizing the training data (overfitting) and helps it generalize better to unseen text.


In [21]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Pop the true labels out of the inputs
        labels = inputs.pop("labels")

        # 2. Feed the text to RoBERTa to get its predictions (logits)
        outputs = model(**inputs)
        logits = outputs.logits

        # 3. Define our Custom Loss Function using our class weights
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)

        # 4. Flatten everything to align it mathematically
        # Use -100 to find and ignore all padding tokens
        active_loss = labels.view(-1) != -100
        active_logits = logits.view(-1, self.model.config.num_labels)[active_loss]
        active_labels = labels.view(-1)[active_loss]

        # 5. Calculate the weighted loss
        loss = loss_fct(active_logits, active_labels)

        return (loss, outputs) if return_outputs else loss

# Initialize seqeval metric
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    # Convert probability scores into exact class predictions
    predictions = np.argmax(predictions, axis=2)

    # Remove the padding (-100) and convert Integer IDs back to String Labels (e.g., 'B-Slogans')
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Let seqeval calculate F1 based on entire BIO spans, not just individual words
    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

The Hyperparameters:

`learning_rate=2e-5 (0.00002)`: Because RoBERTa is already incredibly smart, we want to update its knowledge gently. If the learning rate is too high, the model "forgets" the English language. 2e-5 is the gold standard for fine-tuning NLP models.

`per_device_train_batch_size=16`: The model will look at 16 sentences at a time, calculate its errors, and update its weights. This size is sutable for the capabilities of Google Colab.

`num_train_epochs=3`: The model will read the entire training dataset front-to-back exactly 3 times. For complex tasks like 29-label sequence tagging, going beyond 3 or 4 epochs usually leads to overfitting, which means the model is memorizing the training data instead of learning the actual patterns.

`weight_decay=0.01`: A regularization technique that slightly shrinks the model's weights during training. This prevents the model from memorizing the training data (overfitting) and helps it generalize better to unseen text.

`eval_strategy & save_strategy="epoch"`: At the end of every epoch, the model pauses, takes a test on the 10% Validation set, prints the F1 score, and saves a checkpoint of its brain to your Google Drive.

`load_best_model_at_end=True & metric_for_best_model="f1"`: Often, a model might peak at Epoch 2 and actually get worse in Epoch 3. This setting ensures that when training finishes, Hugging Face deletes the inferior versions and keeps the checkpoint that achieved the highest F1 score on the validation data.

`weight_decay=0.01`: A regularization technique that slightly shrinks the model's weights during training. This prevents the model from memorizing the training data (overfitting) and helps it generalize better to unseen text.

In [24]:
# 1. Initialize Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
).to(device)

# 2. Data Collator for dynamic padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 3. Setup output directory in your Google Drive to save the trained model
output_directory = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/results_exp1'
os.makedirs(output_directory, exist_ok=True)

# 4. Define Training Hyperparameters
training_args = TrainingArguments(
    output_dir=output_directory,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 5. Initialize our Custom Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 6. Start the training
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,2.321911,2.353166,0.004251,0.093537,0.008132,0.463539
2,1.861559,2.046554,0.006292,0.117347,0.011944,0.478761
3,1.539599,2.020922,0.007212,0.127551,0.013652,0.526348


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=2.0093516356556127, metrics={'train_runtime': 632.9523, 'train_samples_per_second': 56.981, 'train_steps_per_second': 3.564, 'total_flos': 1618921298721804.0, 'train_loss': 2.0093516356556127, 'epoch': 3.0})